# 08. SegResNet용 통합 준비 최종 판정

04~07의 승인 기록, split, 전처리 완료 수와 회귀검사 결과를 모아 이진 통합 데이터가 기준선 모델 실험을 시작할 준비가 되었는지 판정합니다. G11 domain shift와 G12 모델 비열등성은 학습 실험 단계이므로 별도 PENDING으로 유지합니다.

In [6]:
from pathlib import Path
import json
import sys

import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'project_config.json').is_file() and (candidate / 'ctprep_utils.py').is_file():
            return candidate
    raise FileNotFoundError('project_config.json이 있는 프로젝트 폴더를 여세요.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from ctqa_utils import as_bool, write_json

RESULTS = PROJECT_ROOT / 'results'
policy = json.loads((RESULTS / '04_integration_policy.json').read_text(encoding='utf-8'))
registry = pd.read_csv(RESULTS / '04_case_review_registry.csv').fillna('')
manifest = pd.read_csv(RESULTS / '05_integration_manifest.csv')
processed_path = RESULTS / '06_preprocessing_manifest.csv'
processed = pd.read_csv(processed_path) if processed_path.is_file() else pd.DataFrame()
regression_path = RESULTS / '07_preprocessing_gate.json'
regression = json.loads(regression_path.read_text(encoding='utf-8')) if regression_path.is_file() else {}

## 1. 남은 조건 계산

In [7]:
allowed = {
    'INTENSITY_SIGNED12': {'APPLY_SIGNED12', 'KEEP_CONFIRMED', 'EXCLUDE'},
    'AFFINE_MISMATCH': {'COPY_CT_AFFINE_TO_MASK', 'RESAMPLE_MASK_TO_CT', 'EXCLUDE'},
    'Z_SPACING_OUTLIER': {'KEEP_CONFIRMED', 'EXCLUDE'},
}
registry['complete_review'] = registry.apply(
    lambda row: as_bool(row.approved)
    and row.action in allowed[row.issue]
    and str(row.reviewer).strip() != ''
    and str(row.rationale).strip() != '',
    axis=1,
)

def issue_resolved(issue):
    subset = registry.loc[registry.issue == issue]
    return bool(len(subset) == 0 or subset.complete_review.all())

active = manifest.loc[manifest.eligible_for_preprocessing.map(as_bool)]
expected_keys = set(zip(active.source, active.case_id))
processed_keys = set(zip(processed.source, processed.case_id)) if len(processed) else set()
processed_complete = expected_keys == processed_keys
split_overlap = (
    manifest.loc[manifest.split.isin(['train', 'validation', 'test'])]
    .groupby(['source', 'group_id']).split.nunique().gt(1).sum()
)
regression_pass = bool(regression.get('regression_pass', False))
license_pass = as_bool(policy.get('license_approved', False)) and bool(str(policy.get('license_evidence', '')).strip())

metrics = {
    'license_pass': license_pass,
    'unresolved_reviews': int((~registry.complete_review).sum()),
    'intensity_review_resolved': issue_resolved('INTENSITY_SIGNED12'),
    'affine_review_resolved': issue_resolved('AFFINE_MISMATCH'),
    'z_spacing_review_resolved': issue_resolved('Z_SPACING_OUTLIER'),
    'eligible_cases': len(expected_keys),
    'processed_cases': len(processed_keys),
    'processed_complete': processed_complete,
    'split_group_overlap': int(split_overlap),
    'regression_pass': regression_pass,
}
print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "license_pass": false,
  "unresolved_reviews": 0,
  "intensity_review_resolved": true,
  "affine_review_resolved": true,
  "z_spacing_review_resolved": true,
  "eligible_cases": 266,
  "processed_cases": 266,
  "processed_complete": true,
  "split_group_overlap": 0,
  "regression_pass": false
}


## 2. 갱신된 통합 게이트

In [8]:
def resolved_gate(resolved):
    if not resolved:
        return 'REVIEW'
    return 'PASS' if processed_complete and regression_pass else 'PENDING'

previous = pd.read_csv(RESULTS / '03_integration_gates.csv')
status_map = {
    'G1 권한·라이선스': 'PASS' if license_pass else 'REVIEW',
    'G4 값 복원': resolved_gate(metrics['intensity_review_resolved']),
    'G5 공간 정합': resolved_gate(metrics['affine_review_resolved']),
    'G6 이상치': resolved_gate(metrics['z_spacing_review_resolved']),
    'G9 split 누수': 'PASS' if split_overlap == 0 else 'FAIL',
    'G10 변환 회귀': 'PASS' if processed_complete and regression_pass else 'PENDING',
}
gates = previous.copy()
# pandas 3.x는 문자열 dtype 열에 정수를 직접 대입하지 못하므로 result를 명시적으로 문자열로 유지합니다.
gates['result'] = gates['result'].astype('string')
gates['status'] = gates.apply(lambda row: status_map.get(row.gate, row.status), axis=1)
gates.loc[gates.gate == 'G1 권한·라이선스', 'result'] = policy.get('license_evidence', '') or '승인 근거 미입력'
gates.loc[gates.gate == 'G4 값 복원', 'result'] = f"미해결 검토={int((registry.issue.eq('INTENSITY_SIGNED12') & ~registry.complete_review).sum())}"
gates.loc[gates.gate == 'G5 공간 정합', 'result'] = f"미해결 검토={int((registry.issue.eq('AFFINE_MISMATCH') & ~registry.complete_review).sum())}"
gates.loc[gates.gate == 'G6 이상치', 'result'] = f"미해결 검토={int((registry.issue.eq('Z_SPACING_OUTLIER') & ~registry.complete_review).sum())}"
gates.loc[gates.gate == 'G9 split 누수', 'result'] = str(int(split_overlap))
gates.loc[gates.gate == 'G10 변환 회귀', 'result'] = f"processed={len(processed_keys)}/{len(expected_keys)}, regression_pass={regression_pass}"
gates.to_csv(RESULTS / '08_integration_gates.csv', index=False, encoding='utf-8-sig')
display(gates[['gate', 'result', 'status']])

,gate,result,status
0,G1 권한·라이선스,승인 근거 미입력,REVIEW
1,G2 파일 인벤토리,0,PASS
2,G3 무결성,0,PASS
3,G4 값 복원,미해결 검토=0,PENDING
4,G5 공간 정합,미해결 검토=0,PENDING
5,G6 이상치,미해결 검토=0,PENDING
6,G7 라벨 일관성,0,PASS
7,G8 ontology,binary: mask>0 가능 / 5종 voxel target 불가,PASS_BINARY_ONLY
8,G9 split 누수,0,PASS
9,G10 변환 회귀,"processed=266/266, regression_pass=False",PENDING


## 3. 최종 판정

G1~G10을 통과하면 통합 데이터로 기준선 실험을 시작할 수 있습니다. 이것이 mixed 모델의 안전성까지 입증했다는 뜻은 아닙니다. G11과 G12는 모델 실험에서 이어서 평가합니다.

In [9]:
required_gates = gates.loc[gates.gate.str.extract(r'G(\d+)')[0].astype(int).le(10)]
acceptable = {'PASS', 'PASS_BINARY_ONLY'}
data_ready = bool(required_gates.status.isin(acceptable).all())
decision = {
    'binary_integration_data_ready': data_ready,
    'binary_training_status': 'READY_FOR_BASELINE_EXPERIMENTS' if data_ready else 'HOLD',
    'five_class_voxel_integration': 'NOT_COMPATIBLE',
    'domain_shift_gate_G11': gates.loc[gates.gate.str.startswith('G11'), 'status'].iloc[0],
    'model_noninferiority_gate_G12': gates.loc[gates.gate.str.startswith('G12'), 'status'].iloc[0],
    'data_blocking_gates': required_gates.loc[~required_gates.status.isin(acceptable), 'gate'].tolist(),
    'remaining_model_gates': gates.loc[gates.gate.str.startswith(('G11', 'G12')), 'gate'].tolist(),
}
write_json(RESULTS / '08_final_readiness.json', decision)
print(json.dumps(decision, ensure_ascii=False, indent=2))

{
  "binary_integration_data_ready": false,
  "binary_training_status": "HOLD",
  "five_class_voxel_integration": "NOT_COMPATIBLE",
  "domain_shift_gate_G11": "PENDING",
  "model_noninferiority_gate_G12": "PENDING",
  "data_blocking_gates": [
    "G1 권한·라이선스",
    "G4 값 복원",
    "G5 공간 정합",
    "G6 이상치",
    "G10 변환 회귀"
  ],
  "remaining_model_gates": [
    "G11 domain shift",
    "G12 모델 비열등성"
  ]
}
